# Camada Gold — Corpus VoC para RAG

**Tech Challenge Fase 4 — Voice of Customer Intelligence com RAG**

Objetivo: Construir o corpus de conhecimento para o pipeline RAG a partir das tabelas Silver.

**Regras de negocio:**
- 1 review com texto = 1 documento
- Inclusao: review com titulo e/ou mensagem nao vazios (~42k docs)
- Exclusao: reviews apenas com nota, sem texto
- document_text = concatenacao de titulo + mensagem
- Metadados: score, datas, status do pedido, atraso, categorias, localizacao

**Fluxo:** Silver (S3) → Gold (S3)

In [0]:
import io
import os
import json
import time
import boto3
import pandas as pd
from dotenv import load_dotenv

# Carrega credenciais do arquivo .env
load_dotenv("/Workspace/Users/justinofilipe03@gmail.com/Tech4/.env")

# Credenciais AWS
AWS_ACCESS_KEY = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-2")

# Bucket e caminhos no S3
S3_BUCKET = os.environ.get("S3_BUCKET", "amzn-s3-fiap-tech-4")
SILVER_PREFIX = os.environ.get("SILVER_PREFIX", "db_silver/")
GOLD_PREFIX = os.environ.get("GOLD_PREFIX", "db_gold/")

# Tabelas Silver necessarias para o Gold
SILVER_TABLES = [
    "olist_order_reviews_dataset",
    "olist_orders_dataset",
    "olist_customers_dataset",
    "olist_order_items_dataset",
    "olist_products_dataset",
    "product_category_name_translation",
]

In [0]:
def get_s3_client():
    """Cria cliente S3 com as credenciais configuradas."""
    return boto3.client(
        "s3",
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        region_name=AWS_REGION,
    )


def read_s3_parquet(s3, prefix, table_name):
    """Le um parquet do S3 e retorna o DataFrame."""
    key = f"{prefix}{table_name}.parquet"
    resp = s3.get_object(Bucket=S3_BUCKET, Key=key)
    df = pd.read_parquet(io.BytesIO(resp["Body"].read()))
    print(f"  [LEITURA] {table_name}: {len(df):,} linhas x {len(df.columns)} colunas")
    return df


def write_s3_parquet(s3, df, key):
    """Grava um DataFrame como parquet no S3."""
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    buffer.seek(0)
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=buffer.getvalue())
    print(f"  [ESCRITA] s3://{S3_BUCKET}/{key}: {len(df):,} linhas")


def write_s3_jsonl(s3, df, key):
    """Grava um DataFrame como JSONL (uma linha por documento)."""
    lines = []
    for _, row in df.iterrows():
        record = {}
        for col in df.columns:
            val = row[col]
            if pd.isna(val):
                record[col] = None
            elif isinstance(val, pd.Timestamp):
                record[col] = val.isoformat()
            else:
                record[col] = val
        lines.append(json.dumps(record, ensure_ascii=False))
    body = "\n".join(lines).encode("utf-8")
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=body)
    print(f"  [ESCRITA] s3://{S3_BUCKET}/{key}: {len(lines):,} documentos")

In [0]:
# ============================================================
# 1. LEITURA DAS TABELAS SILVER
# ============================================================

print("=" * 80)
print("1. LENDO TABELAS SILVER DO S3")
print("=" * 80 + "\n")

s3 = get_s3_client()
silver = {}

for table in SILVER_TABLES:
    silver[table] = read_s3_parquet(s3, SILVER_PREFIX, table)

In [0]:
# ============================================================
# 2. FILTRO — APENAS REVIEWS COM TEXTO
# ============================================================

print("\n" + "=" * 80)
print("2. FILTRANDO REVIEWS COM TEXTO")
print("=" * 80 + "\n")

reviews = silver["olist_order_reviews_dataset"].copy()
total_reviews = len(reviews)

# Filtra: tem titulo OU mensagem (has_title | has_message = True)
reviews = reviews[reviews["has_title"] | reviews["has_message"]].copy()
reviews_with_text = len(reviews)

print(f"  Total de reviews na Silver: {total_reviews:,}")
print(f"  Reviews com texto (Gold): {reviews_with_text:,}")
print(f"  Reviews excluidas (sem texto): {total_reviews - reviews_with_text:,}")

In [0]:
# ============================================================
# 3. CONSTRUCAO DO document_text
# ============================================================

print("\n" + "=" * 80)
print("3. CONSTRUINDO document_text")
print("=" * 80 + "\n")


def build_document_text(row):
    """
    Concatena titulo + mensagem em um unico texto.
    Se ambos existirem: 'titulo. mensagem'
    Se so titulo: titulo
    Se so mensagem: mensagem
    """
    title = row.get("review_comment_title")
    message = row.get("review_comment_message")

    if pd.notna(title) and pd.notna(message):
        return f"{title}. {message}"
    elif pd.notna(title):
        return str(title)
    elif pd.notna(message):
        return str(message)
    return None


reviews["document_text"] = reviews.apply(build_document_text, axis=1)
reviews["text_length"] = reviews["document_text"].str.len()

# Remove eventuais textos vazios apos construcao
reviews = reviews[reviews["document_text"].notna() & (reviews["text_length"] > 0)].copy()

print(f"  document_text criado para {len(reviews):,} reviews")
print(f"  text_length — Min: {reviews['text_length'].min()} | Max: {reviews['text_length'].max()} | Media: {reviews['text_length'].mean():.0f}")

In [0]:
# ============================================================
# 4. AGREGACAO DE CATEGORIAS POR PEDIDO
# ============================================================
# Um pedido pode ter varios itens com categorias diferentes.
# Agregamos as categorias em uma string unica por order_id
# para evitar explosao de linhas no join com reviews.

print("\n" + "=" * 80)
print("4. AGRGANDO CATEGORIAS DE PRODUTO POR PEDIDO")
print("=" * 80 + "\n")

items = silver["olist_order_items_dataset"].copy()
products = silver["olist_products_dataset"].copy()
translation = silver["product_category_name_translation"].copy()

# Join items + products para obter categoria
cat_df = items[["order_id", "product_id"]].merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Produtos sem categoria recebem 'sem_categoria'
cat_df["product_category_name"] = cat_df["product_category_name"].fillna("sem_categoria")

# Join com traducao para nome em ingles
cat_df = cat_df.merge(
    translation,
    on="product_category_name",
    how="left"
)
# Categorias sem traducao mantem o nome em PT
cat_df["product_category_name_english"] = cat_df["product_category_name_english"].fillna(
    cat_df["product_category_name"]
)

# Agrega categorias unicas por order_id (string separada por virgula)
categories_per_order = cat_df.groupby("order_id").agg(
    product_categories_pt=("product_category_name", lambda x: ", ".join(sorted(set(x)))),
    product_categories_en=("product_category_name_english", lambda x: ", ".join(sorted(set(x)))),
    num_categories=("product_category_name", lambda x: len(set(x))),
).reset_index()

print(f"  Pedidos com categorias agregadas: {len(categories_per_order):,}")
print(f"  Categorias unicas por pedido — Min: {categories_per_order['num_categories'].min()} | Max: {categories_per_order['num_categories'].max()} | Media: {categories_per_order['num_categories'].mean():.1f}")

In [0]:
# ============================================================
# 5. MONTAGEM DAS TABELAS GOLD (4 TABELAS INTERLIGADAS)
# ============================================================
# Em vez de uma tabela unica denormalizada, criamos 4 tabelas
# separadas e interligadas por chaves (FK). Isso permite que
# o RAG consulte metadados sob demanda sem carregar tudo.

print("\n" + "=" * 80)
print("5. MONTANDO TABELAS GOLD (4 TABELAS INTERLIGADAS)")
print("=" * 80 + "\n")

# --- Tabela 1: gold_reviews (corpus principal do RAG) ---
# Apenas reviews com texto + derivacoes. FK para gold_orders via order_id.
gold_reviews = reviews[[
    "review_id", "order_id",
    "document_text", "text_length",
    "review_score", "has_title", "has_message",
    "review_creation_date", "review_answer_timestamp",
]].copy()

# Sentimento: 1-2=negative, 3=neutral, 4-5=positive
gold_reviews["sentiment"] = gold_reviews["review_score"].map({
    1: "negative", 2: "negative", 3: "neutral", 4: "positive", 5: "positive",
})

# Ano e mes para filtros temporais
gold_reviews["review_year"] = gold_reviews["review_answer_timestamp"].dt.year
gold_reviews["review_month"] = gold_reviews["review_answer_timestamp"].dt.month

print(f"  gold_reviews: {len(gold_reviews):,} linhas x {len(gold_reviews.columns)} colunas")
print(f"    PK: review_id | FK: order_id -> gold_orders")

# --- Tabela 2: gold_orders (contexto do pedido) ---
# Dados de entrega e status. FK para gold_customers via customer_id.
orders = silver["olist_orders_dataset"].copy()
gold_orders = orders[[
    "order_id", "customer_id", "order_status",
    "order_purchase_timestamp", "delivery_delay_days", "is_late_delivery",
]].copy()

print(f"  gold_orders: {len(gold_orders):,} linhas x {len(gold_orders.columns)} colunas")
print(f"    PK: order_id | FK: customer_id -> gold_customers")

# --- Tabela 3: gold_customers (contexto do cliente) ---
customers = silver["olist_customers_dataset"].copy()
gold_customers = customers[[
    "customer_id", "customer_state", "customer_city",
]].copy()

print(f"  gold_customers: {len(gold_customers):,} linhas x {len(gold_customers.columns)} colunas")
print(f"    PK: customer_id")

# --- Tabela 4: gold_products_by_order (categorias por pedido) ---
# Categorias agregadas por order_id. FK para gold_reviews via order_id.
gold_products_by_order = categories_per_order.copy()

print(f"  gold_products_by_order: {len(gold_products_by_order):,} linhas x {len(gold_products_by_order.columns)} colunas")
print(f"    PK: order_id")

print("\n  Relacionamentos:")
print("    gold_reviews.order_id          -> gold_orders.order_id")
print("    gold_orders.customer_id        -> gold_customers.customer_id")
print("    gold_reviews.order_id          -> gold_products_by_order.order_id")

In [0]:
# ============================================================
# 6. VALIDACAO DE QUALIDADE (4 TABELAS)
# ============================================================

print("\n" + "=" * 80)
print("6. VALIDACAO DE QUALIDADE")
print("=" * 80 + "\n")

checks = []

# gold_reviews
empty_text = gold_reviews["document_text"].isna().sum() + (gold_reviews["document_text"] == "").sum()
checks.append(("gold_reviews: nenhum document_text vazio", empty_text == 0, empty_text))

dup_ids = gold_reviews["review_id"].duplicated().sum()
checks.append(("gold_reviews: review_id unico", dup_ids == 0, dup_ids))

invalid_score = (~gold_reviews["review_score"].between(1, 5)).sum()
checks.append(("gold_reviews: review_score entre 1 e 5", invalid_score == 0, invalid_score))

null_sentiment = gold_reviews["sentiment"].isna().sum()
checks.append(("gold_reviews: sentiment preenchido", null_sentiment == 0, null_sentiment))

# gold_orders
dup_orders = gold_orders["order_id"].duplicated().sum()
checks.append(("gold_orders: order_id unico", dup_orders == 0, dup_orders))

null_status = gold_orders["order_status"].isna().sum()
checks.append(("gold_orders: order_status preenchido", null_status == 0, null_status))

# gold_customers
dup_cust = gold_customers["customer_id"].duplicated().sum()
checks.append(("gold_customers: customer_id unico", dup_cust == 0, dup_cust))

# gold_products_by_order
dup_prod = gold_products_by_order["order_id"].duplicated().sum()
checks.append(("gold_products_by_order: order_id unico", dup_prod == 0, dup_prod))

for name, passed, fail_count in checks:
    status = "OK" if passed else "FALHOU"
    print(f"  [{status}] {name}" + (f" ({fail_count} falhas)" if not passed else ""))

all_passed = all(c[1] for c in checks)
print(f"\n  Resultado: {'TODOS OS CHECKS PASSARAM' if all_passed else 'ALGUNS CHECKS FALHARAM'}")

In [0]:
# ============================================================
# 7. EXPORTACAO PARA S3 (4 PARQUETS INTERLIGADOS)
# ============================================================

print("\n" + "=" * 80)
print("7. EXPORTANDO TABELAS GOLD PARA S3")
print("=" * 80 + "\n")

tables = {
    "gold_reviews": gold_reviews,
    "gold_orders": gold_orders,
    "gold_customers": gold_customers,
    "gold_products_by_order": gold_products_by_order,
}

for name, df in tables.items():
    write_s3_parquet(s3, df, f"{GOLD_PREFIX}{name}.parquet")

print("\nExportacao concluida!")

In [0]:
# ============================================================
# 8. ESQUEMA RELACIONAL DAS TABELAS GOLD
# ============================================================

print("\n" + "=" * 80)
print("8. ESQUEMA RELACIONAL — TABELAS GOLD")
print("=" * 80)

print("""
  gold_reviews (PK: review_id, FK: order_id -> gold_orders)
  |__ review_id              string    — PK, para citacao no RAG
  |__ order_id               string    — FK para gold_orders
  |__ document_text          string    — texto para embedding (titulo + mensagem)
  |__ text_length            int       — comprimento do document_text
  |__ review_score           int       — nota 1-5
  |__ sentiment              string    — positive / neutral / negative
  |__ has_title              bool      — tem titulo?
  |__ has_message            bool      — tem mensagem?
  |__ review_creation_date   datetime  — data de criacao
  |__ review_answer_timestamp datetime — data de resposta
  |__ review_year            int       — ano (para filtro temporal)
  |__ review_month           int       — mes (para filtro temporal)

  gold_orders (PK: order_id, FK: customer_id -> gold_customers)
  |__ order_id               string    — PK
  |__ customer_id            string    — FK para gold_customers
  |__ order_status           string    — delivered / canceled / shipped / etc.
  |__ order_purchase_timestamp datetime — data da compra
  |__ delivery_delay_days    float     — atraso em dias (negativo = adiantado)
  |__ is_late_delivery       bool      — atrasou?

  gold_customers (PK: customer_id)
  |__ customer_id            string    — PK
  |__ customer_state         string    — UF (SP, RJ, MG, etc.)
  |__ customer_city          string    — cidade

  gold_products_by_order (PK: order_id)
  |__ order_id               string    — PK, FK para gold_reviews
  |__ product_categories_pt  string    — categorias em PT (separadas por virgula)
  |__ product_categories_en  string    — categorias em EN (separadas por virgula)
  |__ num_categories         int       — numero de categorias distintas

  RELACIONAMENTOS:
  gold_reviews.order_id          -> gold_orders.order_id
  gold_orders.customer_id        -> gold_customers.customer_id
  gold_reviews.order_id          -> gold_products_by_order.order_id
""")

In [0]:
# ============================================================
# 9. RELATORIO FINAL DAS TABELAS GOLD
# ============================================================

print("\n" + "=" * 80)
print("RELATORIO FINAL — CAMADA GOLD (4 TABELAS INTERLIGADAS)")
print("=" * 80)

print(f"\n{'Tabela':<30} {'Linhas':>8} {'Cols':>5} {'Chave':>12}")
print("-" * 58)
print(f"{'gold_reviews':<30} {len(gold_reviews):>8,} {len(gold_reviews.columns):>5} {'review_id':>12}")
print(f"{'gold_orders':<30} {len(gold_orders):>8,} {len(gold_orders.columns):>5} {'order_id':>12}")
print(f"{'gold_customers':<30} {len(gold_customers):>8,} {len(gold_customers.columns):>5} {'customer_id':>12}")
print(f"{'gold_products_by_order':<30} {len(gold_products_by_order):>8,} {len(gold_products_by_order.columns):>5} {'order_id':>12}")
print("-" * 58)

print(f"\n{'Distribuicao por sentimento (gold_reviews)':<45} {'Docs':>8} {'%':>7}")
print("-" * 62)
for s in ["positive", "neutral", "negative"]:
    count = (gold_reviews["sentiment"] == s).sum()
    print(f"  {s:<43} {count:>8,} {count/len(gold_reviews)*100:>6.1f}%")

print(f"\n{'Cobertura de metadados (via joins)':<45} {'Preenchido':>10} {'%':>7}")
print("-" * 62)
# Simula o join para verificar cobertura
temp = gold_reviews.merge(gold_orders, on="order_id", how="left")
temp = temp.merge(gold_customers, on="customer_id", how="left")
temp = temp.merge(gold_products_by_order, on="order_id", how="left")

meta_cols = ["order_status", "delivery_delay_days", "customer_state", "product_categories_pt"]
for col in meta_cols:
    filled = temp[col].notna().sum()
    print(f"  {col:<43} {filled:>10,} {filled/len(temp)*100:>6.1f}%")

print(f"\n{'Caracteristicas do texto (gold_reviews)':<45} {'Valor':>15}")
print("-" * 62)
print(f"  {'text_length min':<43} {gold_reviews['text_length'].min():>15}")
print(f"  {'text_length max':<43} {gold_reviews['text_length'].max():>15}")
print(f"  {'text_length media':<43} {gold_reviews['text_length'].mean():>15.0f}")
print(f"  {'text_length mediana':<43} {gold_reviews['text_length'].median():>15.0f}")

print("\n" + "=" * 80)
print("CAMADA GOLD CONCLUIDA — 4 TABELAS INTERLIGADAS NO S3")
print("=" * 80)

In [0]:
# Pipeline Gold executado com sucesso.
# 4 tabelas .parquet interligadas disponiveis em s3://{S3_BUCKET}/{GOLD_PREFIX}
#
# gold_reviews.parquet          — corpus principal (reviews com texto)
# gold_orders.parquet          — contexto de pedidos (status, entrega)
# gold_customers.parquet       — contexto de clientes (estado, cidade)
# gold_products_by_order.parquet — categorias de produtos por pedido

print("Pipeline Gold executado com sucesso.")
print(f"4 tabelas disponiveis em s3://{S3_BUCKET}/{GOLD_PREFIX}")